In [2]:
import pandas as pd
import numpy as np
import os
import re
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

In [3]:
df = pd.read_csv("crop_yield.csv")

df.head()

,Crop,Crop_Year,Season,State,Area,Production,Annual_Rainfall,Fertilizer,Pesticide,Yield
0,Arecanut,1997,Whole Year,Assam,73814.0,56708,2051.4,7024878.38,22882.34,0.796087
1,Arhar/Tur,1997,Kharif,Assam,6637.0,4685,2051.4,631643.29,2057.47,0.710435
2,Castor seed,1997,Kharif,Assam,796.0,22,2051.4,75755.32,246.76,0.238333
3,Coconut,1997,Whole Year,Assam,19656.0,126905000,2051.4,1870661.52,6093.36,5238.051739
4,Cotton(lint),1997,Kharif,Assam,1739.0,794,2051.4,165500.63,539.09,0.420909


In [4]:
df.columns = df.columns.str.strip().str.lower()
df.columns

Index(['crop', 'crop_year', 'season', 'state', 'area', 'production',
       'annual_rainfall', 'fertilizer', 'pesticide', 'yield'],
      dtype='object')

In [5]:
df = df.drop(columns=["production"])

In [6]:
df.shape

(19689, 9)

In [7]:
df.info()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19689 entries, 0 to 19688
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   crop             19689 non-null  object 
 1   crop_year        19689 non-null  int64  
 2   season           19689 non-null  object 
 3   state            19689 non-null  object 
 4   area             19689 non-null  float64
 5   annual_rainfall  19689 non-null  float64
 6   fertilizer       19689 non-null  float64
 7   pesticide        19689 non-null  float64
 8   yield            19689 non-null  float64
dtypes: float64(5), int64(1), object(3)
memory usage: 1.4+ MB


In [8]:
df.describe()

,crop_year,area,annual_rainfall,fertilizer,pesticide,yield
count,19689.000000,1.968900e+04,19689.000000,1.968900e+04,1.968900e+04,19689.000000
mean,2009.127584,1.799266e+05,1437.755177,2.410331e+07,4.884835e+04,79.954009
std,6.498099,7.328287e+05,816.909589,9.494600e+07,2.132874e+05,878.306193
min,1997.000000,5.000000e-01,301.300000,5.417000e+01,9.000000e-02,0.000000
25%,2004.000000,1.390000e+03,940.700000,1.880146e+05,3.567000e+02,0.600000
50%,2010.000000,9.317000e+03,1247.600000,1.234957e+06,2.421900e+03,1.030000
75%,2015.000000,7.511200e+04,1643.700000,1.000385e+07,2.004170e+04,2.388889
max,2020.000000,5.080810e+07,6552.700000,4.835407e+09,1.575051e+07,21105.000000


In [9]:
df.isnull().sum()

crop               0
crop_year          0
season             0
state              0
area               0
annual_rainfall    0
fertilizer         0
pesticide          0
yield              0
dtype: int64

In [10]:
df["crop"] = df["crop"].str.strip()
df["season"] = df["season"].str.strip()
df["state"] = df["state"].str.strip()

In [12]:
df["crop"].unique()

array(['Arecanut', 'Arhar/Tur', 'Castor seed', 'Coconut', 'Cotton(lint)',
       'Dry chillies', 'Gram', 'Jute', 'Linseed', 'Maize', 'Mesta',
       'Niger seed', 'Onion', 'Other  Rabi pulses', 'Potato',
       'Rapeseed &Mustard', 'Rice', 'Sesamum', 'Small millets',
       'Sugarcane', 'Sweet potato', 'Tapioca', 'Tobacco', 'Turmeric',
       'Wheat', 'Bajra', 'Black pepper', 'Cardamom', 'Coriander',
       'Garlic', 'Ginger', 'Groundnut', 'Horse-gram', 'Jowar', 'Ragi',
       'Cashewnut', 'Banana', 'Soyabean', 'Barley', 'Khesari', 'Masoor',
       'Moong(Green Gram)', 'Other Kharif pulses', 'Safflower',
       'Sannhamp', 'Sunflower', 'Urad', 'Peas & beans (Pulses)',
       'other oilseeds', 'Other Cereals', 'Cowpea(Lobia)',
       'Oilseeds total', 'Guar seed', 'Other Summer Pulses', 'Moth'],
      dtype=object)

In [13]:
df["season"].unique()

array(['Whole Year', 'Kharif', 'Rabi', 'Autumn', 'Summer', 'Winter'],
      dtype=object)

In [14]:
df["state"].unique()

array(['Assam', 'Karnataka', 'Kerala', 'Meghalaya', 'West Bengal',
       'Puducherry', 'Goa', 'Andhra Pradesh', 'Tamil Nadu', 'Odisha',
       'Bihar', 'Gujarat', 'Madhya Pradesh', 'Maharashtra', 'Mizoram',
       'Punjab', 'Uttar Pradesh', 'Haryana', 'Himachal Pradesh',
       'Tripura', 'Nagaland', 'Chhattisgarh', 'Uttarakhand', 'Jharkhand',
       'Delhi', 'Manipur', 'Jammu and Kashmir', 'Telangana',
       'Arunachal Pradesh', 'Sikkim'], dtype=object)

In [16]:
X = df[
[
    "crop",
    "crop_year",
    "season",
    "state",
    "area",
    "annual_rainfall",
    "fertilizer",
    "pesticide"
]]

y = df["yield"]

In [17]:
cat_cols = ["crop", "season", "state"]

num_cols = [
    "crop_year",
    "area",
    "annual_rainfall",
    "fertilizer",
    "pesticide"
]

In [18]:
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
])

In [19]:
model = Pipeline([
    ("prep", preprocessor),
    ("rf", RandomForestRegressor(
        n_estimators=250,
        random_state=42,
        max_depth=20
    ))
])

In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [21]:
model.fit(X_train, y_train)

,steps,"[('prep', ...), ('rf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [22]:
pred = model.predict(X_test)

r2 = r2_score(y_test, pred)
mae = mean_absolute_error(y_test, pred)

print("R2 Score:", r2)
print("MAE:", mae)

R2 Score: 0.9797271671568929
MAE: 9.605189393808976


In [23]:
joblib.dump(model, "yield_model.pkl")
print("Model Saved")

Model Saved


In [24]:
sample = pd.DataFrame({
    "crop":["Wheat"],
    "crop_year":[2020],
    "season":["Rabi"],
    "state":["Punjab"],
    "area":[2],
    "annual_rainfall":[120],
    "fertilizer":[80],
    "pesticide":[20]
})

prediction = model.predict(sample)
print(prediction[0])

3.258273825436454


In [27]:
results_df = pd.DataFrame(results, columns=["Crop", "R2", "MAE"])
results_df.sort_values("R2", ascending=False)

,Crop,R2,MAE
25,Bajra,0.951570,0.437282
19,Sugarcane,0.939957,5.549516
28,Coriander,0.913574,0.156772
21,Tapioca,0.903772,1.898273
38,Barley,0.876521,0.232186
44,Sannhamp,0.872310,0.278665
31,Groundnut,0.865854,0.146780
29,Garlic,0.865595,0.896434
3,Coconut,0.865560,954.117776
2,Castor seed,0.864474,0.117402


In [28]:
model = joblib.load("models/wheat.pkl")

sample = pd.DataFrame({
    "crop_year":[2020],
    "season":["Rabi"],
    "state":["Punjab"],
    "area":[2],
    "annual_rainfall":[120],
    "fertilizer":[80],
    "pesticide":[20]
})

prediction = model.predict(sample)
prediction[0]

np.float64(2.884063420379998)

In [29]:
import joblib

model = joblib.load("models/wheat.pkl")

In [31]:
import numpy as np

# numeric features (same order as training)
numeric_cols = [
    "crop_year",
    "area",
    "annual_rainfall",
    "fertilizer",
    "pesticide"
]

# get encoded categorical feature names
cat_features = model.named_steps["prep"]\
    .named_transformers_["cat"]\
    .get_feature_names_out(["season","state"])

# combine all feature names
feature_names = list(numeric_cols) + list(cat_features)